# B5 silhouette 진단과 개선 실험

기존 B5와 B4 결과를 보존합니다. 원본 태그 특징과 군집 좌표를 구분하고, SVD 후 L2 정규화를 적용합니다.
축소 공간의 점수 상승만으로 채택하지 않습니다. 후보 선택에는 고정된 C01 원본 공간·같은 책 표본의 silhouette를 사용합니다.


In [1]:
from pathlib import Path
from dataclasses import asdict
from time import perf_counter
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import sklearn
from threadpoolctl import threadpool_limits
from sklearn.metrics import adjusted_rand_score
from bookrec.data import load_book_tags, load_books
from bookrec.models.content_based.tuning import ContentBasedConfig, make_content_based
from bookrec.models.content_based.clustering import (
    prepare_cluster_space, fit_book_clusters, cluster_diagnostics, cluster_overview, representative_books,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "results" / "content_based"
OUT.mkdir(parents=True, exist_ok=True)
old_files = [p for p in OUT.glob("B5_*") if not p.name.startswith("B5_improvement")]
old_files += [ROOT / "notebooks/content_based/B5_book_clustering.ipynb", OUT / "B4_selected_config.json"]
old_files += [p for p in (ROOT / "docs/figures").glob("B5_*.png") if not p.name.startswith("B5_improvement")]
old_hashes = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in old_files}
b4 = json.loads((OUT / "B4_selected_config.json").read_text())
base_config = ContentBasedConfig(**b4["config"])
tags = load_book_tags()
books = load_books()
reference = make_content_based(tags, base_config).features
SELECTION_SEED, CONFIRMATION_SEED = 42, 2026
SAMPLE_SIZE, MIN_BOOKS = 2000, 100
SVD_SEED = 42
DIMENSIONS = (None, 50, 100, 200)
K_CANDIDATES = (5, 8, 10, 12, 15, 20)
FIT_SEEDS = (42, 7, 2026)
MIN_GAIN = 0.0001
sample_ids = np.sort(np.random.default_rng(SELECTION_SEED).choice(reference.book_ids, size=SAMPLE_SIZE, replace=False))
# NOTE: 원래 B5는 행 위치 0~9999에서 표본을 뽑았다. ID 1~10000에서 같은 난수로 뽑아 동일한 표본을 얻는다.
confirmation_ids = np.sort(np.random.default_rng(CONFIRMATION_SEED).choice(reference.book_ids, size=SAMPLE_SIZE, replace=False))
print("원본 기준 특징:", reference.matrix.shape)
print("독자 평점·test·cold_final은 사용하지 않음")


원본 기준 특징: (10000, 17042)
독자 평점·test·cold_final은 사용하지 않음


## 1. 평가 기준과 실행 함수

- k=10에서 축소 없음·50·100·200차원을 먼저 비교합니다.
- 모든 후보는 같은 원본 기준 공간(C01), Euclidean 거리, 고정 2,000권 표본으로 비교합니다.
- clustering_silhouette는 후보의 실제 군집 공간 점수입니다. 공간이 달라지므로 기준선 점수와 직접 비교하지 않습니다.
- original_silhouette는 각 후보의 축소 전 특징 공간, reference_silhouette는 항상 같은 C01 공간입니다.
- 최소 군집 100권, 미배정 0권 조건으로 기준 공간 점수를 선택합니다. 차이가 0.0001 이하이면 기존 결과를 우선합니다.
- SVD가 원본 기준을 개선하지 못하면 basic/extended, binary/TF-IDF를 원본 및 가장 유망한 SVD 차원으로 비교합니다.
- 선택한 표현은 k 6개 x 학습 시드 3개로 확인합니다. 별도 표본은 선택에 사용하지 않고 마지막에 확인합니다.

SVD는 seed=42, n_iter=7입니다. 군집은 기존과 같은 n_init=10, max_iter=300, tol=1e-4, Lloyd입니다.
네이티브 수치 연산 스레드를 2개로 제한해 과도한 병렬 실행을 피합니다. 실제 노트북은 위에서부터 재실행할 수 있습니다.


In [2]:
records = []

def run_candidate(candidate_id, stage, features, config, space, k=10, fit_seed=42):
    start = perf_counter()
    with threadpool_limits(limits=2):
        clusters = fit_book_clusters(features, k, seed=fit_seed, space=space)
        per_book, per_cluster = cluster_diagnostics(features, clusters, sample_ids, reference_features=reference)
    sizes = np.bincount(clusters.labels[clusters.labels >= 0], minlength=k)
    row = {"candidate_id": candidate_id, "stage": stage, **asdict(config),
           "n_components": space.n_components or 0, "svd_seed": SVD_SEED, "fit_seed": fit_seed, "k": k,
           "explained_variance_ratio": space.explained_variance_ratio,
           "features": features.matrix.shape[1], "inertia": clusters.inertia, "n_iter": clusters.n_iter,
           "min_books": int(sizes.min()), "max_books": int(sizes.max()),
           "unassigned_books": int((clusters.labels < 0).sum()), "sample_books": len(per_book),
           "eligible": bool(sizes.min() >= MIN_BOOKS and (clusters.labels >= 0).all()),
           "seconds_fit_diagnostics": perf_counter() - start}
    for name in ("original", "clustering", "reference"):
        row[f"{name}_silhouette"] = float(per_book[f"{name}_silhouette"].mean())
        row[f"{name}_negative_rate"] = float((per_book[f"{name}_silhouette"] < 0).mean())
    records.append(row)
    pd.DataFrame(records).to_csv(OUT / "B5_improvement_comparison.csv", index=False)
    print(f"{candidate_id}: reference={row['reference_silhouette']:.6f}, cluster-space={row['clustering_silhouette']:.6f}, min={sizes.min()}, {row['seconds_fit_diagnostics']:.1f}초")
    return clusters, per_book, per_cluster


## 2. 기존 결과 진단과 SVD 비교

기존 k=10의 배정·silhouette가 재현되는지 확인합니다. 낮은 silhouette 책은 오류로 단정하지 않고, 겹치는 군집의 사례로 노트북에 출력합니다.


In [3]:
spaces = {}
for dimension in DIMENSIONS:
    start = perf_counter()
    with threadpool_limits(limits=2):
        spaces[dimension] = prepare_cluster_space(reference, dimension, seed=SVD_SEED)
    print(f"좌표 생성: {dimension or '원본'}, {perf_counter() - start:.2f}초")
    candidate_id = f"svd_{dimension}" if dimension else "baseline"
    result, per_book, per_cluster = run_candidate(candidate_id, "dimensions", reference, base_config, spaces[dimension])
    if dimension is None:
        baseline = result
        baseline_row = records[-1].copy()
        saved_assignments = pd.read_csv(OUT / "B5_assignments.csv")
        np.testing.assert_array_equal(result.book_ids, saved_assignments["book_id"])
        np.testing.assert_array_equal(result.labels, saved_assignments["cluster"])
        expected = json.loads((OUT / "B5_selected_clustering.json").read_text())["silhouette"]
        np.testing.assert_allclose(baseline_row["reference_silhouette"], expected, rtol=0, atol=1e-12)
        baseline_diagnostics = per_cluster.copy()
        boundary = per_book.nsmallest(30, "reference_silhouette").merge(books[["book_id", "title"]], on="book_id", validate="one_to_one")
        print(boundary.head(10).to_string(index=False))
        print(per_cluster.to_string(index=False))
print(pd.DataFrame(records)[["candidate_id", "reference_silhouette", "clustering_silhouette", "reference_negative_rate"]].to_string(index=False))


좌표 생성: 원본, 0.02초
baseline: reference=0.062676, cluster-space=0.062676, min=393, 0.9초
 book_id  cluster  original_silhouette  clustering_silhouette  reference_silhouette                                           title
    7837        3            -0.040940              -0.040940             -0.040940                            Twilight and History
    8802        3            -0.040284              -0.040284             -0.040284                      The City of Falling Angels
    5788        3            -0.034360              -0.034360             -0.034360 ABNKKBSNPLAKo?! (Mga Kwentong Chalk ni Bob Ong)
    8393        0            -0.031544              -0.031544             -0.031544       Enemy of God (The Warlord Chronicles, #2)
    6895        0            -0.028506              -0.028506             -0.028506                               How We Are Hungry
    2379        3            -0.027620              -0.027620             -0.027620                               My Friend

## 3. 필요한 경우 태그 표현 비교

SVD만으로 공통 기준 점수가 개선되지 않은 경우에만 B1~B4를 변경하지 않고 다른 특징 설정을 비교합니다.


In [4]:
stage_one = pd.DataFrame(records)
svd_rows = stage_one[(stage_one["candidate_id"] != "baseline") & stage_one["eligible"]]
best_svd = svd_rows.sort_values(["reference_silhouette", "n_components"], ascending=[False, True]).iloc[0]
feature_search_performed = best_svd["reference_silhouette"] <= baseline_row["reference_silhouette"] + MIN_GAIN
feature_options = {"baseline": (reference, base_config)}
if feature_search_performed:
    for extended, weighting in [(False, "tfidf"), (True, "binary"), (True, "tfidf")]:
        config = ContentBasedConfig(extended=extended, weighting=weighting)
        features = make_content_based(tags, config).features
        key = f"{'extended' if extended else 'basic'}_{weighting}"
        feature_options[key] = (features, config)
        for dimension in (None, int(best_svd["n_components"])):
            with threadpool_limits(limits=2):
                space = prepare_cluster_space(features, dimension, seed=SVD_SEED)
            run_candidate(f"{key}_{dimension or 0}", "features", features, config, space)
else:
    print("SVD가 고정 원본 기준을 개선해 추가 태그 탐색은 생략합니다.")
comparison = pd.DataFrame(records)
initial = comparison[comparison["eligible"]].sort_values(
    ["reference_silhouette", "n_components", "candidate_id"], ascending=[False, True, True]).iloc[0]
if initial["reference_silhouette"] <= baseline_row["reference_silhouette"] + MIN_GAIN:
    initial = comparison[comparison["candidate_id"] == "baseline"].iloc[0]
chosen_config = ContentBasedConfig(**{key: initial[key] for key in asdict(base_config)})
# NOTE: pandas의 bool/int 스칼라를 원래 Python 형으로 되돌려 설정 생성 검증과 JSON 저장을 맞춘다.
chosen_config = ContentBasedConfig(extended=bool(chosen_config.extended), weighting=str(chosen_config.weighting),
                                  min_df=int(chosen_config.min_df), sublinear_tf=bool(chosen_config.sublinear_tf),
                                  profile_weighting=str(chosen_config.profile_weighting))
chosen_dimension = int(initial["n_components"]) or None
chosen_features = reference if chosen_config == base_config else make_content_based(tags, chosen_config).features
with threadpool_limits(limits=2):
    chosen_space = prepare_cluster_space(chosen_features, chosen_dimension, seed=SVD_SEED)
print("안정성 확인 대상:", asdict(chosen_config), "차원:", chosen_dimension)


SVD가 고정 원본 기준을 개선해 추가 태그 탐색은 생략합니다.
안정성 확인 대상: {'extended': False, 'weighting': 'binary', 'min_df': 2, 'sublinear_tf': True, 'profile_weighting': 'positive'} 차원: 50


## 4. k와 초기화 안정성

SVD 좌표는 고정하고 k-means 시드만 42·7·2026으로 바꿉니다. k마다 공통 기준 silhouette 평균으로 비교합니다.
모든 시드에서 최소 100권·미배정 0권을 만족해야 합니다. 시드 42의 군집을 배포 결과로 고정해 최고 시드만 고르지 않습니다.
기존 원본 k=10도 같은 세 시드로 비교합니다. ARI는 같은 k의 시드 42 결과와의 배정 일치도를 나타냅니다.


In [5]:
stability_labels = {}
for k in K_CANDIDATES:
    for seed in FIT_SEEDS:
        name = f"stability_k{k}_seed{seed}"
        result, _, _ = run_candidate(name, "stability", chosen_features, chosen_config, chosen_space, k=k, fit_seed=seed)
        stability_labels[(k, seed)] = result.labels
# NOTE: 원본 표현이 선택됐다면 같은 실행을 재활용한다.
if chosen_config == base_config and chosen_dimension is None:
    baseline_stability = [dict(row) for row in records if row["stage"] == "stability" and row["k"] == 10]
else:
    baseline_stability = []
    for seed in FIT_SEEDS:
        result, _, _ = run_candidate(f"baseline_seed{seed}", "baseline_stability", reference, base_config, spaces[None], fit_seed=seed)
        baseline_stability.append(records[-1].copy())
stability = pd.DataFrame([row for row in records if row["stage"] == "stability"])
stability["ari_vs_seed42"] = [adjusted_rand_score(stability_labels[(row.k, 42)], stability_labels[(row.k, row.fit_seed)]) for row in stability.itertuples()]
summary = stability.groupby("k").agg(reference_mean=("reference_silhouette", "mean"),
    reference_std=("reference_silhouette", "std"), clustering_mean=("clustering_silhouette", "mean"),
    minimum_books=("min_books", "min"), eligible=("eligible", "all"), mean_ari_vs_seed42=("ari_vs_seed42", "mean")).reset_index()
best = summary[summary["eligible"]].sort_values(["reference_mean", "k"], ascending=[False, True]).iloc[0]
proposed_k = int(best["k"])
print(summary.to_string(index=False))
print("원본 k=10의 시드 평균:", np.mean([r["reference_silhouette"] for r in baseline_stability]))


stability_k5_seed42: reference=0.055223, cluster-space=0.144674, min=1084, 0.1초
stability_k5_seed7: reference=0.055223, cluster-space=0.144674, min=1084, 0.1초
stability_k5_seed2026: reference=0.055223, cluster-space=0.144674, min=1084, 0.1초
stability_k8_seed42: reference=0.056385, cluster-space=0.156899, min=545, 0.1초
stability_k8_seed7: reference=0.056346, cluster-space=0.156862, min=545, 0.1초
stability_k8_seed2026: reference=0.060092, cluster-space=0.163495, min=394, 0.1초
stability_k10_seed42: reference=0.063772, cluster-space=0.178420, min=389, 0.2초
stability_k10_seed7: reference=0.062946, cluster-space=0.176599, min=388, 0.2초
stability_k10_seed2026: reference=0.062948, cluster-space=0.176568, min=388, 0.2초
stability_k12_seed42: reference=0.063284, cluster-space=0.184735, min=356, 0.2초
stability_k12_seed7: reference=0.058583, cluster-space=0.171007, min=385, 0.2초
stability_k12_seed2026: reference=0.059585, cluster-space=0.172687, min=384, 0.1초
stability_k15_seed42: reference=0.05811

## 5. 별도 표본 확인과 채택 판단

다른 seed=2026의 2,000권은 여기서 처음 평가합니다. 선택 표본과 일부 책이 겹칠 수 있으며 독립적인 일반화 평가를 뜻하지 않습니다.


In [6]:
with threadpool_limits(limits=2):
    proposed = fit_book_clusters(chosen_features, proposed_k, seed=42, space=chosen_space)
    proposed_selection_books, _ = cluster_diagnostics(chosen_features, proposed, sample_ids, reference_features=reference)
    proposed_confirmation, _ = cluster_diagnostics(chosen_features, proposed, confirmation_ids, reference_features=reference)
    baseline_confirmation, _ = cluster_diagnostics(reference, baseline, confirmation_ids)
baseline_mean = float(np.mean([r["reference_silhouette"] for r in baseline_stability]))
confirmation_gain = float(proposed_confirmation.reference_silhouette.mean() - baseline_confirmation.reference_silhouette.mean())
seed42_gain = float(proposed_selection_books.reference_silhouette.mean() - baseline_row["reference_silhouette"])
adopted = bool(best["reference_mean"] > baseline_mean + MIN_GAIN and seed42_gain > MIN_GAIN and confirmation_gain > MIN_GAIN)
final_clusters = proposed if adopted else baseline
final_features = chosen_features if adopted else reference
final_config = chosen_config if adopted else base_config
final_dimension = chosen_dimension if adopted else None
final_k = proposed_k if adopted else 10
confirmation = pd.DataFrame([
    {"candidate_id": "baseline", "sample": "confirmation", "sample_seed": CONFIRMATION_SEED,
     "reference_silhouette": float(baseline_confirmation.reference_silhouette.mean())},
    {"candidate_id": "proposed", "sample": "confirmation", "sample_seed": CONFIRMATION_SEED,
     "reference_silhouette": float(proposed_confirmation.reference_silhouette.mean())},
])
print(confirmation.to_string(index=False))
print("기준 공간 평균 개선:", float(best["reference_mean"] - baseline_mean))
print("seed42 선택 표본 개선:", seed42_gain, "별도 표본 개선:", confirmation_gain, "채택:", adopted)


candidate_id       sample  sample_seed  reference_silhouette
    baseline confirmation         2026              0.062817
    proposed confirmation         2026              0.065548
기준 공간 평균 개선: 0.0008065474262029665
seed42 선택 표본 개선: 0.0010964374779517394 별도 표본 개선: 0.0027314743817101195 채택: True


## 6. 최종 산출물과 기존 결과 보존

채택하지 않으면 기존 군집을 개선 파일에도 명시적으로 남깁니다. 태그 설명은 원래 태그 평균으로, 대표 책 거리는 실제 군집 좌표로 구합니다.
SVD의 음수 좌표는 정상이며 태그 이름과 직접 대응시키지 않습니다.


In [7]:
overview = cluster_overview(final_features, final_clusters)
representatives = representative_books(final_features, final_clusters, n_books=5).merge(
    books[["book_id", "title", "main_author"]], on="book_id", validate="many_to_one")
assignments = pd.DataFrame({"book_id": final_clusters.book_ids, "cluster": final_clusters.labels})
with threadpool_limits(limits=2):
    final_sample, final_diagnostics = cluster_diagnostics(final_features, final_clusters, sample_ids, reference_features=reference)
# NOTE: 군집 설명과 최종 진단을 대표 책 표에 합쳐 별도 요약 CSV를 만들지 않는다.
cluster_details = overview.rename(columns={"books": "cluster_books"}).merge(
    final_diagnostics[["cluster", "sample_books", "reference_silhouette", "reference_negative_rate",
                       "clustering_silhouette", "clustering_negative_rate"]], on="cluster", validate="one_to_one")
representatives = representatives.merge(cluster_details, on="cluster", validate="many_to_one")
representatives.to_csv(OUT / "B5_improvement_representative_books.csv", index=False)
assignments.to_csv(OUT / "B5_improvement_assignments.csv", index=False)
# NOTE: 시드별 ARI와 별도 표본 결과도 전체 비교표 하나에 모은다. stage와 sample_seed로 구분한다.
comparison = pd.DataFrame(records)
comparison["sample_seed"] = SELECTION_SEED
comparison["ari_vs_seed42"] = comparison["candidate_id"].map(stability.set_index("candidate_id")["ari_vs_seed42"])
confirmation_rows = []
proposed_row = next(row for row in records if row["candidate_id"] == f"stability_k{proposed_k}_seed42")
for name, template, scores in [("baseline", baseline_row, baseline_confirmation),
                                ("proposed", proposed_row, proposed_confirmation)]:
    row = {**template, "candidate_id": f"confirmation_{name}", "stage": "confirmation",
           "sample_seed": CONFIRMATION_SEED, "seconds_fit_diagnostics": np.nan, "ari_vs_seed42": np.nan}
    for metric in ("original", "clustering", "reference"):
        row[f"{metric}_silhouette"] = float(scores[f"{metric}_silhouette"].mean())
        row[f"{metric}_negative_rate"] = float((scores[f"{metric}_silhouette"] < 0).mean())
    confirmation_rows.append(row)
comparison = pd.concat([comparison, pd.DataFrame(confirmation_rows)], ignore_index=True)
comparison.to_csv(OUT / "B5_improvement_comparison.csv", index=False)
settings = {
    "adopted": adopted, "feature_config": asdict(final_config), "n_components": final_dimension,
    "k": final_k, "svd_seed": SVD_SEED, "svd_n_iter": 7, "fit_seed": 42,
    "n_init": 10, "max_iter": 300, "tol": 1e-4, "algorithm": "lloyd", "initialization": "k-means++",
    "sample_size": SAMPLE_SIZE, "selection_sample_seed": SELECTION_SEED, "confirmation_sample_seed": CONFIRMATION_SEED,
    "min_cluster_books": MIN_BOOKS, "minimum_gain": MIN_GAIN, "metric": "euclidean",
    "criterion": "fixed C01 reference silhouette; all 3 seeds meet size/coverage; mean and seed42 selection gain and confirmation gain > 0.0001",
    "reference_config": asdict(base_config), "feature_search_performed": bool(feature_search_performed),
    "reference_silhouette": float(final_sample.reference_silhouette.mean()),
    "clustering_silhouette": float(final_sample.clustering_silhouette.mean()),
    "proposed_k": proposed_k, "proposed_n_components": chosen_dimension,
    "proposed_reference_mean": float(best["reference_mean"]), "baseline_reference_mean": baseline_mean,
    "proposed_confirmation_gain": confirmation_gain, "reference_space_changed": False,
    "explained_variance_ratio": final_clusters.space.explained_variance_ratio,
    "python": platform.python_version(), "numpy": np.__version__, "sklearn": sklearn.__version__,
}
(OUT / "B5_improvement_selected_config.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2) + "\n")
assert assignments.book_id.is_unique and len(assignments) == 10000
assert overview.books.sum() == 10000 and len(representatives) == final_k * 5
assert representatives.book_id.is_unique and representatives.title.notna().all()
for cluster, group in representatives.groupby("cluster"):
    labels = assignments.set_index("book_id").cluster.loc[group.book_id]
    assert labels.eq(cluster).all() and group.distance.is_monotonic_increasing
for name, digest in old_hashes.items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest, name
print("기존 B5·B4 파일 해시 보존 확인 통과")
print(json.dumps(settings, ensure_ascii=False, indent=2))
print(overview.to_string(index=False))
print(representatives[representatives["rank"] == 1].to_string(index=False))


기존 B5·B4 파일 해시 보존 확인 통과
{
  "adopted": true,
  "feature_config": {
    "extended": false,
    "weighting": "binary",
    "min_df": 2,
    "sublinear_tf": true,
    "profile_weighting": "positive"
  },
  "n_components": 50,
  "k": 10,
  "svd_seed": 42,
  "svd_n_iter": 7,
  "fit_seed": 42,
  "n_init": 10,
  "max_iter": 300,
  "tol": 0.0001,
  "algorithm": "lloyd",
  "initialization": "k-means++",
  "sample_size": 2000,
  "selection_sample_seed": 42,
  "confirmation_sample_seed": 2026,
  "min_cluster_books": 100,
  "minimum_gain": 0.0001,
  "metric": "euclidean",
  "criterion": "fixed C01 reference silhouette; all 3 seeds meet size/coverage; mean and seed42 selection gain and confirmation gain > 0.0001",
  "reference_config": {
    "extended": false,
    "weighting": "binary",
    "min_df": 2,
    "sublinear_tf": true,
    "profile_weighting": "positive"
  },
  "feature_search_performed": false,
  "reference_silhouette": 0.0637722130983365,
  "clustering_silhouette": 0.1784200880213679,
 

## 7. 그래프

차원 비교와 k·시드 비교 그림 2개만 저장합니다. 공용 setup과 save를 사용하고 평가 공간을 구분합니다. 군집별 진단은 노트북 출력과 대표 책 CSV에서 확인합니다.


In [8]:
import os
import tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "bookrec-matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from bookrec.plot_style import setup, save, MODEL_COLORS

setup()
dimensions = pd.DataFrame([r for r in records if r["stage"] == "dimensions"])
fig, ax = plt.subplots()
x = np.arange(len(dimensions))
ax.bar(x - 0.18, dimensions["reference_silhouette"], width=0.36, label="Fixed original space", color=MODEL_COLORS["cb"])
ax.bar(x + 0.18, dimensions["clustering_silhouette"], width=0.36, label="Clustering space", color=MODEL_COLORS["mf"])
ax.set_xticks(x, ["Original", "SVD 50", "SVD 100", "SVD 200"])
ax.set_ylabel("Silhouette (Euclidean, same 2,000 books)")
ax.set_title("B5: separating representation change from improvement")
ax.legend()
save(fig, "B5_improvement_dimension_comparison")
plt.close(fig)
fig, ax = plt.subplots()
ax.errorbar(summary["k"], summary["reference_mean"], yerr=summary["reference_std"], marker="o", capsize=4,
            label="Chosen representation: mean +/- SD (3 seeds)", color=MODEL_COLORS["cb"])
ax.axhline(baseline_mean, linestyle="--", color=MODEL_COLORS["baseline"], label="Original k=10: 3-seed mean")
ax.set_xticks(K_CANDIDATES)
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette in fixed original space")
ax.set_title("B5: cluster-count and initialization comparison")
ax.legend(fontsize=9)
save(fig, "B5_improvement_k_stability")
plt.close(fig)
print("공용 save로 그래프 2개 저장")


공용 save로 그래프 2개 저장


## 해석

축소 공간 점수와 원래 공간 점수를 구분합니다. 낮은 점수를 가진 책을 삭제하거나 군집 크기 조건을 완화해 점수를 올리지 않습니다.
SVD는 태그 의미의 정답을 학습하지 않습니다. 최종 태그·대표 책을 보고 사람이 해석해야 합니다.
별도 표본은 같은 카탈로그에서 추출한 감도 확인이며 독립적인 테스트셋은 아닙니다. B4 추천 설정과 기존 B5 파일은 유지됩니다.


## 저장 파일

노트북 외에 결과 파일 5개와 그래프 2개만 저장합니다. 표본 ID는 설정의 seed로 재생성하며, 중간 진단은 노트북 출력에서 확인합니다.

- B5_improvement_comparison.csv: 설정별 점수·실행 시간, 안정성 단계의 ARI, 별도 표본 결과. stage와 sample_seed로 구분합니다.
- B5_improvement_selected_config.json: 최종 설정과 선택 기준.
- B5_improvement_assignments.csv: 10,000권의 군집 배정.
- B5_improvement_representative_books.csv: 대표 책 50권과 군집별 책 수·태그·진단을 합친 표. 군집 정보는 대표 책마다 반복됩니다.
- B5_improvement.md: 실험 설명과 재현 방법.
- docs/figures/B5_improvement_dimension_comparison.png: 차원 비교.
- docs/figures/B5_improvement_k_stability.png: k·시드 비교.
